In [22]:
import cv2
import gradio as gr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import hog

### Feature Extraction Functions

In [23]:
def extract_spatial_features(img, size=(16, 16)):
    return cv2.resize(img, size).ravel()

In [31]:
def extract_color_hist(img, nbins=32):
    ch1 = np.histogram(img[:, :, 0], bins=nbins, range=(0, 256))[0]
    ch2 = np.histogram(img[:, :, 1], bins=nbins, range=(0, 256))[0]
    ch3 = np.histogram(img[:, :, 2], bins=nbins, range=(0, 256))[0]
    return np.concatenate((ch1, ch2, ch3))

In [32]:
def extract_hog_features(img_gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)):
    return hog(
        img_gray,
        orientations=orientations,
        pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block,
        transform_sqrt=True,
        feature_vector=True,
        visualize=False
    )

In [33]:
def extract_all_feature_components(img_path):
    
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None

    img_ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    spatial_feat = extract_spatial_features(img_ycrcb)
    hist_feat = extract_color_hist(img_ycrcb)
    hog_feat = extract_hog_features(img_gray)

    feature_vector = np.concatenate((spatial_feat, hist_feat, hog_feat))

    return {
        "spatial_features": spatial_feat.tolist(),
        "hist_features": hist_feat.tolist(),
        "hog_features": hog_feat.tolist(),
        "feature_vector": feature_vector.tolist()
    }

### Build Dataset Pipeline

In [35]:
import os
import glob

In [36]:
data_dir = '../../dataset/car detection/color'

In [ ]:
def build_csv_dataset(data_dir, output_csv="vehicle_dataset.csv"):

    print("[1/3] Scanning image paths from dataset folder...")
    records = []

    veh_paths = glob.glob(os.path.join(data_dir, "vehicles", "**", "*.png"), recursive=True)
    if not veh_paths:
        veh_paths = glob.glob(os.path.join(data_dir, "vehicles", "**", "*.jpg"), recursive=True)
        
    for p in veh_paths:
        records.append({
            "image_id": os.path.splitext(os.path.basename(p))[0],
            "file_path": p,
            "label": 1,
            "label_name": "vehicle"
        })

    non_veh_paths = glob.glob(os.path.join(data_dir, "non-vehicles", "**", "*.png"), recursive=True)
    if not non_veh_paths:
        non_veh_paths = glob.glob(os.path.join(data_dir, "non-vehicles", "**", "*.jpg"), recursive=True)
        
    for p in non_veh_paths:
        records.append({
            "image_id": os.path.splitext(os.path.basename(p))[0],
            "file_path": p,
            "label": 0,
            "label_name": "non_vehicle"
        })

    df = pd.DataFrame(records)

    print("\n[2/3] Extracting features and populating feature columns...")

    spatial_col, hist_col, hog_col, full_vector_col = [], [], [], []
    valid_indices = []

    for idx, row in df.iterrows():
        features_dict = extract_all_feature_components(row['file_path'])
        if features_dict is not None:
            spatial_col.append(features_dict["spatial_features"])
            hist_col.append(features_dict["hist_features"])
            hog_col.append(features_dict["hog_features"])
            full_vector_col.append(features_dict["feature_vector"])
            valid_indices.append(idx)
        else:
            print(f" -> Warning: Failed to read image at {row['file_path']}")

    df = df.loc[valid_indices].reset_index(drop=True)

    # Assign specific feature columns directly to DataFrame
    df["spatial_features"] = spatial_col
    df["hist_features"] = hist_col
    df["hog_features"] = hog_col
    df["feature_vector"] = full_vector_col

    print(f"\n[3/3] Exporting dataset with explicit feature columns to {output_csv}...")
    df.to_csv(output_csv, index=False)
    print(f" -> Successfully saved! CSV shape: {df.shape}")



In [38]:
build_csv_dataset(data_dir)


[3/3] Exporting dataset with explicit feature columns to vehicle_dataset.csv...
 -> Successfully saved! CSV shape: (17760, 8)
